# CatBoost 모델 구성 설명

## 목표와 Target

- 목표: 학생별로 **해당 예측 주차 종료 시점에서 다음 주 이탈확률**을 예측합니다.
- 사용 데이터: `ML/used_data/weekly_next_week_with_vle_enhanced.csv`
- Target: `target_next_week_withdrawn` — 다음 주 이탈이면 1, 아니면 0

## 사용 Feature

모델 입력은 `id_student`와 Target을 제외한 **124개 Feature**입니다. `id_student`는 모델 입력에 넣지 않고 GroupKFold 분할 기준으로만 사용합니다.

- 과목·운영·주차: `code_module`, `code_presentation`, `prediction_week`, `cutoff_day`, 강좌 운영 길이
- 학생 특성: 성별, 지역, 최종 학력, IMD 구간, 연령대, 이전 수강 횟수, 수강 학점, 장애 여부
- 등록 Feature: 등록일, 개강 후 등록 여부, 사전 등록일 수, 지연 등록일 수
- 평가 누적 Feature: 평가 수·비중·제출 수·지각 제출 수·미제출 수·제출률·평균 점수
- VLE 누적 Feature: 누적 클릭 수·활동일 수·고유 사이트 수·마지막 활동일·활동 유형별 누적 클릭 수·VLE 기록 여부

모든 평가·VLE Feature는 `prediction_week` 종료일까지 누적된 값만 사용하며, 미래 정보와 이탈 후 행은 제외되어 있습니다.

## CatBoost 전처리·검증

- 범주형 8개(`code_module`, `code_presentation`, 성별·지역·학력·IMD·연령·장애 여부)는 문자열로 변환한 뒤 CatBoost에 범주형 열로 직접 전달합니다. **One-Hot Encoding은 사용하지 않습니다.**
- 수치형 Feature는 그대로 사용하고 무한값은 결측값으로 바꿉니다.
- `id_student` 기준 **3-Fold GroupKFold**로 분할해 동일 학생의 여러 주차 행이 Train과 Test에 동시에 들어가지 않게 합니다.

## 하이퍼파라미터

| 항목 | 설정값 | 의미 |
|---|---:|---|
| `loss_function` | `Logloss` | 이진 이탈확률 손실함수 |
| `eval_metric` | `PRAUC` | 불균형 데이터용 PR-AUC 기준 |
| `iterations` | 500 | 최대 부스팅 반복 수 |
| `learning_rate` | 0.05 | 작은 학습률로 점진적 학습 |
| `depth` | 7 | 트리 깊이 |
| `l2_leaf_reg` | 5 | L2 규제 강도 |
| `od_type` | `Iter` | 반복 기준 조기 종료 방식 |
| `od_wait` | 40 | 검증 PR-AUC 개선이 없을 때 기다리는 반복 수 |
| `random_seed` | 42 | 재현성 확보 |
| `use_best_model` | `True` | 검증 성능이 가장 좋은 반복 수의 모델 사용 |

현재 CatBoost 코드에는 별도 클래스 가중치를 적용하지 않았습니다. 따라서 원시 확률의 보정 상태를 함께 평가할 수 있으며, 필요하면 최종 선택 후 Platt Scaling 또는 Isotonic Regression으로 추가 보정을 검증합니다.

## 학습·예측·평가 흐름

각 Fold에서 Train 데이터로 CatBoost를 학습한 뒤, Test Fold의 다음 주 이탈확률을 예측합니다. 세 Fold의 Test 예측을 합친 OOF 확률로 `Recall@Top-20%`, `PR-AUC`, `Brier Score`, `ECE(10구간)`를 계산합니다.

# 확장 데이터 재학습 결과

- 사용 데이터: `ML/used_data/weekly_next_week_with_vle_enhanced.csv`
- 데이터 크기: 895,005행 × 126열, 모델 입력 Feature 124개
- 검증 방식: `GroupKFold(id_student, 3-Fold)`
- OOF Recall@Top-20%: **0.711331**
- OOF PR-AUC: **0.094775**
- Brier Score: **0.007045**
- ECE(10-bin): **0.000242**

> 이 수치는 확장 데이터로 재학습한 결과이다. CatBoost는 확률 품질(Brier Score, ECE)이 특히 우수한지 함께 확인한다.


# CatBoost: 주차별 다음 주 이탈 예측

`weekly_next_week_with_vle_enhanced.csv`를 사용합니다. CatBoost는 범주형 변수를 직접 처리하며, `id_student` 기준 GroupKFold로 XGBoost와 동일한 기준에서 비교합니다.

In [ ]:
from pathlib import Path
import runpy

current = Path.cwd().resolve()
script_path = next(
    candidate / 'models' / 'ML' / '02_catboost_weekly_next_week.py'
    for candidate in [current, *current.parents]
    if (candidate / 'models' / 'ML' / '02_catboost_weekly_next_week.py').is_file()
)
runpy.run_path(str(script_path), run_name='__main__')

In [ ]:
import pandas as pd
pd.read_csv('catboost_weekly_next_week_metrics.csv')